# Fine-tune YOLO11n Chess Detection (Multi-board + Noise)

**Setup:** Runtime > Change runtime type > **GPU** (T4 free / A100 pro)

Everything is saved to Google Drive — safe from disconnects.

## Workflow
1. Mount Drive + install deps
2. Generate data (or restore from Drive)
3. Fine-tune (saves checkpoints to Drive)
4. Compare old vs new model
5. Export ONNX

---
## 1. Setup

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/chess_training'
DRIVE_DATA = f'{DRIVE_DIR}/datasets_multi.tar.gz'
DRIVE_WEIGHTS = f'{DRIVE_DIR}/multi_board_v2'
LOCAL_DATA = '/content/datasets_multi'

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DRIVE_WEIGHTS, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

In [ ]:
!pip install ultralytics onnxruntime requests -q

In [ ]:
# Clone repo + download base model (v0.0.4)
if not os.path.exists('/content/2d-chess-pieces-detection'):
    !git clone https://github.com/gsdt/2d-chess-pieces-detection.git /content/2d-chess-pieces-detection
%cd /content/2d-chess-pieces-detection

if not os.path.exists('base_model.pt'):
    !wget -q https://github.com/Zai-Kun/2d-chess-pieces-detection/releases/download/v0.0.4/chess_detectionv0.0.4.pt -O base_model.pt
print('Base model ready')

---
## 2. Generate or Restore Data

In [ ]:
RESTORE_FROM_DRIVE = os.path.exists(DRIVE_DATA)

if RESTORE_FROM_DRIVE:
    print(f'Restoring data from Drive: {DRIVE_DATA}')
    !tar xzf {DRIVE_DATA} -C /content/
    import glob
    train_count = len(glob.glob(f'{LOCAL_DATA}/images/train/*.jpg'))
    val_count = len(glob.glob(f'{LOCAL_DATA}/images/val/*.jpg'))
    print(f'Restored: {train_count} train, {val_count} val')
else:
    print('No data on Drive. Will generate fresh data in next cell.')

In [ ]:
%%writefile generate_multi_board.py
"""Generate training data with multiple chess boards per image + noise.

Adds realistic distractions: random text, lines, shapes, noise,
and optionally background images to simulate real-world screenshots
from books, websites, and documents.

Usage:
    # Download background images first (optional but recommended)
    cd assets && python random_images_downloader2.py && cd ..

    # Generate data
    python generate_multi_board.py

Output:
    datasets_multi/images/{train,val}/  - images (1280x1280)
    datasets_multi/labels/{train,val}/  - YOLO labels
    chess_detection_multi.yaml          - dataset config
"""

import os
import random
import string
from PIL import Image, ImageDraw, ImageFont, ImageFilter
from random_fen_gen import generate_fen

# --- Config ---
IMG_SIZE = 1280
BOARD_MIN_SIZE = 120
BOARD_MAX_SIZE = 500
MAX_BOARDS_PER_IMAGE = 12
DATA_SPLIT = 0.8

# Noise config
ADD_TEXT = True
TEXT_PROBABILITY = 0.7
MAX_TEXT_BLOCKS = 8

ADD_LINES = True
LINE_PROBABILITY = 0.5
MAX_LINES = 15

ADD_SHAPES = True
SHAPE_PROBABILITY = 0.4
MAX_SHAPES = 10

ADD_NOISE_DOTS = True
NOISE_PROBABILITY = 0.3
MAX_NOISE_DOTS = 200

ADD_BLUR = True
BLUR_PROBABILITY = 0.1

ADD_FAKE_SQUARES = True
FAKE_SQUARE_PROBABILITY = 0.5
MAX_FAKE_SQUARES = 5

USE_BACKGROUND_IMAGES = True
BACKGROUND_PROBABILITY = 0.5
BACKGROUND_DIR = "assets/random_noise_backgrounds"

BOARDS_DIR = "assets/boards"
PIECES_DIR = "assets/pieces"
DATASETS_DIR = "datasets_multi"

FEN_TO_PIECE = {
    "p": "bP", "r": "bR", "n": "bN", "b": "bB", "q": "bQ", "k": "bK",
    "P": "wP", "R": "wR", "N": "wN", "B": "wB", "Q": "wQ", "K": "wK",
}
PIECE_TO_CLASS = {k: i for i, k in enumerate(FEN_TO_PIECE.keys())}
BOARD_CLASS = 12

# Sample text for noise
CHESS_WORDS = [
    "White to move", "Black to move", "Checkmate in 2", "Diagram",
    "Figure", "Position after", "Exercise", "Solution", "Chapter",
    "1. e4 e5", "2. Nf3 Nc6", "3. Bb5 a6", "Sicilian Defense",
    "Queen's Gambit", "King's Indian", "French Defense", "Caro-Kann",
    "1-0", "0-1", "1/2-1/2", "+=", "=+", "+-", "-+",
    "Analysis by", "Source:", "Rating: 2400", "Page",
    "www.chess.com", "lichess.org", "FIDE", "ELO",
]
LOREM_WORDS = [
    "the", "quick", "brown", "fox", "jumps", "over", "lazy", "dog",
    "chess", "position", "board", "game", "move", "piece", "square",
    "opening", "middlegame", "endgame", "strategy", "tactics",
]


def load_board(name):
    return Image.open(f"{BOARDS_DIR}/{name}").convert("RGB")


def load_pieces(name):
    return {
        fen_char: Image.open(f"{PIECES_DIR}/{name}/{piece_name}.png").convert("RGBA")
        for fen_char, piece_name in FEN_TO_PIECE.items()
    }


def load_backgrounds():
    """Load background images if available."""
    if not os.path.isdir(BACKGROUND_DIR):
        return []
    bgs = []
    for f in os.listdir(BACKGROUND_DIR):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            try:
                bgs.append(Image.open(f"{BACKGROUND_DIR}/{f}").convert("RGB"))
            except:
                pass
    return bgs


def render_board(board_img, piece_set, fen, board_size):
    """Render a chess position onto a board image at given size."""
    board = board_img.copy().resize((board_size, board_size))
    tile = board_size / 8

    for row, rank in enumerate(fen.split("/")):
        col = 0
        for char in rank:
            if char.isdigit():
                col += int(char)
            elif char in piece_set:
                piece = piece_set[char].copy().resize((int(tile), int(tile)))
                x, y = int(col * tile), int(row * tile)
                board.paste(piece, (x, y), piece)
                col += 1
    return board


def get_labels(fen, bx, by, board_size):
    """Generate YOLO labels for one board placed at (bx, by)."""
    labels = []
    tile = board_size / 8

    cx = (bx + board_size / 2) / IMG_SIZE
    cy = (by + board_size / 2) / IMG_SIZE
    w = board_size / IMG_SIZE
    h = board_size / IMG_SIZE
    labels.append(f"{BOARD_CLASS} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")

    for row, rank in enumerate(fen.split("/")):
        col = 0
        for char in rank:
            if char.isdigit():
                col += int(char)
            elif char in PIECE_TO_CLASS:
                px = bx + col * tile + tile / 2
                py = by + row * tile + tile / 2
                labels.append(
                    f"{PIECE_TO_CLASS[char]} {px/IMG_SIZE:.6f} {py/IMG_SIZE:.6f} "
                    f"{tile/IMG_SIZE:.6f} {tile/IMG_SIZE:.6f}"
                )
                col += 1
    return labels


def boxes_overlap(b1, b2):
    """Check if two boxes (x, y, w, h) overlap."""
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    return not (x1 + w1 <= x2 or x2 + w2 <= x1 or y1 + h1 <= y2 or y2 + h2 <= y1)


def place_boards(num_boards):
    """Find non-overlapping positions for multiple boards."""
    placements = []
    gap = 10

    max_size = min(BOARD_MAX_SIZE, IMG_SIZE // max(2, int(num_boards ** 0.5)))
    min_size = max(BOARD_MIN_SIZE, max_size // 3)

    for _ in range(num_boards):
        for _attempt in range(200):
            size = random.randint(min_size, max_size)
            x = random.randint(0, IMG_SIZE - size)
            y = random.randint(0, IMG_SIZE - size)
            box = (x, y, size, size)
            padded = (x - gap, y - gap, size + 2 * gap, size + 2 * gap)
            if not any(boxes_overlap(padded, p) for p in placements):
                placements.append(box)
                break
    return placements


# --- Noise functions ---

def add_random_text(canvas, placements):
    """Add random text blocks that don't overlap with boards."""
    if not ADD_TEXT or random.random() > TEXT_PROBABILITY:
        return canvas
    draw = ImageDraw.Draw(canvas)
    num_blocks = random.randint(1, MAX_TEXT_BLOCKS)

    for _ in range(num_blocks):
        # Random font size
        font_size = random.randint(12, 40)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
        except:
            font = ImageFont.load_default()

        # Random text content
        if random.random() < 0.5:
            text = random.choice(CHESS_WORDS)
        else:
            words = random.sample(LOREM_WORDS, random.randint(2, 6))
            text = " ".join(words)
            if random.random() < 0.3:
                # Multi-line text
                text = "\n".join([" ".join(random.sample(LOREM_WORDS, random.randint(2, 5)))
                                  for _ in range(random.randint(2, 5))])

        # Random position (try to avoid boards)
        for _attempt in range(50):
            x = random.randint(0, IMG_SIZE - 100)
            y = random.randint(0, IMG_SIZE - 50)
            text_box = (x, y, 300, font_size + 10)
            if not any(boxes_overlap(text_box, p) for p in placements):
                break

        # Random color
        color = (random.randint(0, 200), random.randint(0, 200), random.randint(0, 200))
        draw.text((x, y), text, fill=color, font=font)

    return canvas


def add_random_lines(canvas):
    """Add random lines and arrows."""
    if not ADD_LINES or random.random() > LINE_PROBABILITY:
        return canvas
    draw = ImageDraw.Draw(canvas)
    num_lines = random.randint(1, MAX_LINES)

    for _ in range(num_lines):
        x1 = random.randint(0, IMG_SIZE)
        y1 = random.randint(0, IMG_SIZE)
        x2 = random.randint(0, IMG_SIZE)
        y2 = random.randint(0, IMG_SIZE)
        color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
        width = random.randint(1, 5)
        draw.line([(x1, y1), (x2, y2)], fill=color, width=width)

    return canvas


def add_random_shapes(canvas):
    """Add random rectangles, circles, and other shapes."""
    if not ADD_SHAPES or random.random() > SHAPE_PROBABILITY:
        return canvas
    draw = ImageDraw.Draw(canvas)
    num_shapes = random.randint(1, MAX_SHAPES)

    for _ in range(num_shapes):
        x1 = random.randint(0, IMG_SIZE - 50)
        y1 = random.randint(0, IMG_SIZE - 50)
        x2 = x1 + random.randint(20, 200)
        y2 = y1 + random.randint(20, 200)
        color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))

        shape_type = random.choice(['rectangle', 'ellipse', 'rectangle_outline'])
        if shape_type == 'rectangle':
            # Semi-transparent fill
            overlay = Image.new('RGBA', canvas.size, (0, 0, 0, 0))
            overlay_draw = ImageDraw.Draw(overlay)
            alpha = random.randint(30, 120)
            overlay_draw.rectangle([x1, y1, x2, y2], fill=(*color, alpha))
            canvas = Image.alpha_composite(canvas.convert('RGBA'), overlay).convert('RGB')
        elif shape_type == 'ellipse':
            draw = ImageDraw.Draw(canvas)
            draw.ellipse([x1, y1, x2, y2], outline=color, width=random.randint(1, 4))
        else:
            draw = ImageDraw.Draw(canvas)
            draw.rectangle([x1, y1, x2, y2], outline=color, width=random.randint(1, 3))

    return canvas


def add_noise_dots(canvas):
    """Add random noise dots."""
    if not ADD_NOISE_DOTS or random.random() > NOISE_PROBABILITY:
        return canvas
    draw = ImageDraw.Draw(canvas)
    num_dots = random.randint(50, MAX_NOISE_DOTS)

    for _ in range(num_dots):
        x = random.randint(0, IMG_SIZE)
        y = random.randint(0, IMG_SIZE)
        color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
        radius = random.randint(1, 5)
        draw.ellipse([(x, y), (x + radius, y + radius)], fill=color)

    return canvas


def add_blur(canvas):
    """Apply slight blur to the whole image."""
    if not ADD_BLUR or random.random() > BLUR_PROBABILITY:
        return canvas
    radius = random.uniform(0.5, 1.5)
    return canvas.filter(ImageFilter.GaussianBlur(radius=radius))


def add_fake_squares(canvas, placements):
    """Add square-shaped objects that are NOT chess boards.

    Teaches model to distinguish real chess boards from:
    - Colored squares, tables, grids
    - Checkerboard-like patterns (but wrong)
    - Images, icons, UI elements
    - QR-code-like patterns
    """
    if not ADD_FAKE_SQUARES or random.random() > FAKE_SQUARE_PROBABILITY:
        return canvas
    draw = ImageDraw.Draw(canvas)
    num_squares = random.randint(1, MAX_FAKE_SQUARES)

    for _ in range(num_squares):
        size = random.randint(60, 300)

        # Find position that doesn't overlap with real boards
        placed = False
        for _attempt in range(100):
            x = random.randint(0, IMG_SIZE - size)
            y = random.randint(0, IMG_SIZE - size)
            box = (x, y, size, size)
            padded = (x - 5, y - 5, size + 10, size + 10)
            if not any(boxes_overlap(padded, p) for p in placements):
                placed = True
                break
        if not placed:
            continue

        fake_type = random.choice([
            'solid_square', 'bordered_square', 'wrong_grid',
            'checkerboard_wrong', 'color_blocks', 'icon_grid',
            'table', 'photo_placeholder'
        ])

        if fake_type == 'solid_square':
            # Just a colored square
            color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
            draw.rectangle([x, y, x + size, y + size], fill=color)

        elif fake_type == 'bordered_square':
            # Square with thick border
            bg = (random.randint(200, 255), random.randint(200, 255), random.randint(200, 255))
            border = (random.randint(0, 100), random.randint(0, 100), random.randint(0, 100))
            draw.rectangle([x, y, x + size, y + size], fill=bg, outline=border,
                           width=random.randint(2, 6))

        elif fake_type == 'wrong_grid':
            # Grid but NOT 8x8 (e.g. 3x3, 5x5, 6x6, 10x10)
            n = random.choice([3, 4, 5, 6, 7, 9, 10])
            cell = size // n
            colors = [
                ((255, 255, 255), (200, 200, 200)),  # gray
                ((255, 200, 200), (200, 100, 100)),  # red
                ((200, 255, 200), (100, 200, 100)),  # green
                ((200, 200, 255), (100, 100, 200)),  # blue
            ]
            c1, c2 = random.choice(colors)
            for row in range(n):
                for col in range(n):
                    cx, cy = x + col * cell, y + row * cell
                    color = c1 if (row + col) % 2 == 0 else c2
                    draw.rectangle([cx, cy, cx + cell, cy + cell], fill=color)

        elif fake_type == 'checkerboard_wrong':
            # 8x8 but wrong colors / with random content (not chess pieces)
            cell = size // 8
            c1 = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
            c2 = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
            for row in range(8):
                for col in range(8):
                    cx, cy = x + col * cell, y + row * cell
                    color = c1 if (row + col) % 2 == 0 else c2
                    draw.rectangle([cx, cy, cx + cell, cy + cell], fill=color)
                    # Random symbols (not chess pieces)
                    if random.random() < 0.3:
                        char = random.choice('★●▲■◆✕○△□◇+×=#&%')
                        try:
                            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
                                                      cell // 2)
                        except:
                            font = ImageFont.load_default()
                        text_color = (255 - color[0], 255 - color[1], 255 - color[2])
                        draw.text((cx + cell // 4, cy + cell // 4), char,
                                  fill=text_color, font=font)

        elif fake_type == 'color_blocks':
            # Random colored blocks in a grid
            n = random.choice([2, 3, 4, 5, 6])
            cell = size // n
            for row in range(n):
                for col in range(n):
                    cx, cy = x + col * cell, y + row * cell
                    color = (random.randint(50, 255), random.randint(50, 255), random.randint(50, 255))
                    draw.rectangle([cx, cy, cx + cell, cy + cell], fill=color,
                                   outline=(0, 0, 0), width=1)

        elif fake_type == 'icon_grid':
            # Grid of small squares (like app icons)
            n = random.choice([3, 4, 5])
            cell = size // n
            gap = cell // 8
            for row in range(n):
                for col in range(n):
                    cx = x + col * cell + gap
                    cy = y + row * cell + gap
                    s = cell - 2 * gap
                    color = (random.randint(50, 255), random.randint(50, 255), random.randint(50, 255))
                    # Rounded look
                    draw.rectangle([cx, cy, cx + s, cy + s], fill=color)

        elif fake_type == 'table':
            # Table with rows and columns (like spreadsheet)
            rows = random.randint(3, 8)
            cols = random.randint(2, 5)
            cell_w = size // cols
            cell_h = size // rows
            # Header
            draw.rectangle([x, y, x + size, y + cell_h],
                           fill=(random.randint(50, 100), random.randint(80, 130), random.randint(120, 180)))
            # Grid lines
            for row in range(rows + 1):
                draw.line([(x, y + row * cell_h), (x + size, y + row * cell_h)],
                          fill=(150, 150, 150), width=1)
            for col in range(cols + 1):
                draw.line([(x + col * cell_w, y), (x + col * cell_w, y + size)],
                          fill=(150, 150, 150), width=1)
            # Random text in cells
            for row in range(1, rows):
                for col in range(cols):
                    if random.random() < 0.6:
                        text = str(random.randint(0, 999))
                        draw.text((x + col * cell_w + 5, y + row * cell_h + 5),
                                  text, fill=(60, 60, 60))

        elif fake_type == 'photo_placeholder':
            # Gray placeholder with X or icon
            draw.rectangle([x, y, x + size, y + size], fill=(220, 220, 220),
                           outline=(180, 180, 180), width=2)
            # Diagonal cross
            draw.line([(x, y), (x + size, y + size)], fill=(180, 180, 180), width=1)
            draw.line([(x + size, y), (x, y + size)], fill=(180, 180, 180), width=1)
            # "IMAGE" text
            try:
                font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", size // 8)
            except:
                font = ImageFont.load_default()
            draw.text((x + size // 4, y + size // 2 - 10), "IMAGE",
                      fill=(160, 160, 160), font=font)

    return canvas


def add_real_image_crops(canvas, placements, backgrounds):
    """Paste random crops from real photos onto the canvas.

    Simulates web pages with mixed content: photos, ads, thumbnails
    alongside chess boards.
    """
    if not backgrounds or random.random() > 0.5:
        return canvas
    num_crops = random.randint(1, 4)

    for _ in range(num_crops):
        bg = random.choice(backgrounds)
        # Random crop size
        crop_w = random.randint(80, 400)
        crop_h = random.randint(80, 300)

        # Random crop from background image
        bw, bh = bg.size
        if bw <= crop_w or bh <= crop_h:
            continue
        cx = random.randint(0, bw - crop_w)
        cy = random.randint(0, bh - crop_h)
        crop = bg.crop((cx, cy, cx + crop_w, cy + crop_h))

        # Place on canvas (avoid overlapping boards)
        for _attempt in range(50):
            px = random.randint(0, IMG_SIZE - crop_w)
            py = random.randint(0, IMG_SIZE - crop_h)
            box = (px, py, crop_w, crop_h)
            if not any(boxes_overlap(box, p) for p in placements):
                canvas.paste(crop, (px, py))
                break

    return canvas


def random_background(backgrounds):
    """Generate a background - either solid color or from background image."""
    if USE_BACKGROUND_IMAGES and backgrounds and random.random() < BACKGROUND_PROBABILITY:
        bg = random.choice(backgrounds).copy()
        bg = bg.resize((IMG_SIZE, IMG_SIZE))
        # Random brightness adjustment
        from PIL import ImageEnhance
        enhancer = ImageEnhance.Brightness(bg)
        bg = enhancer.enhance(random.uniform(0.7, 1.3))
        return bg

    # Solid or gradient background
    bg = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
    draw = ImageDraw.Draw(bg)

    bg_type = random.choice(['solid', 'gradient_v', 'gradient_h'])
    if bg_type == 'solid':
        r, g, b = random.randint(150, 255), random.randint(150, 255), random.randint(150, 255)
        draw.rectangle([0, 0, IMG_SIZE, IMG_SIZE], fill=(r, g, b))
    elif bg_type == 'gradient_v':
        r1, g1, b1 = random.randint(150, 255), random.randint(150, 255), random.randint(150, 255)
        r2, g2, b2 = random.randint(150, 255), random.randint(150, 255), random.randint(150, 255)
        for y in range(IMG_SIZE):
            t = y / IMG_SIZE
            r = int(r1 + (r2 - r1) * t)
            g = int(g1 + (g2 - g1) * t)
            b = int(b1 + (b2 - b1) * t)
            draw.line([(0, y), (IMG_SIZE, y)], fill=(r, g, b))
    else:
        r1, g1, b1 = random.randint(150, 255), random.randint(150, 255), random.randint(150, 255)
        r2, g2, b2 = random.randint(150, 255), random.randint(150, 255), random.randint(150, 255)
        for x in range(IMG_SIZE):
            t = x / IMG_SIZE
            r = int(r1 + (r2 - r1) * t)
            g = int(g1 + (g2 - g1) * t)
            b = int(b1 + (b2 - b1) * t)
            draw.line([(x, 0), (x, IMG_SIZE)], fill=(r, g, b))

    return bg


def generate_negative_text_only(backgrounds, img_path, label_path):
    """Generate image with ONLY text, no chess board (negative sample)."""
    canvas = random_background(backgrounds)
    draw = ImageDraw.Draw(canvas)

    # Fill with lots of text
    num_blocks = random.randint(5, 20)
    for _ in range(num_blocks):
        font_size = random.randint(10, 50)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
        except:
            font = ImageFont.load_default()

        # Generate paragraph text
        lines = []
        for _ in range(random.randint(1, 8)):
            words = random.sample(LOREM_WORDS + CHESS_WORDS, min(random.randint(3, 10), len(LOREM_WORDS)))
            lines.append(" ".join(words))
        text = "\n".join(lines)

        x = random.randint(0, IMG_SIZE - 200)
        y = random.randint(0, IMG_SIZE - 100)
        color = (random.randint(0, 180), random.randint(0, 180), random.randint(0, 180))
        draw.text((x, y), text, fill=color, font=font)

    canvas = add_random_lines(canvas)
    canvas = add_fake_squares(canvas, [])
    canvas = add_noise_dots(canvas)

    canvas.save(img_path, "JPEG", quality=95)
    with open(label_path, "w") as f:
        f.write("")  # empty label = no objects


def generate_negative_images_only(backgrounds, img_path, label_path):
    """Generate image with ONLY photos/images, no chess board (negative sample)."""
    canvas = random_background(backgrounds)

    if backgrounds:
        # Paste many random image crops
        num_crops = random.randint(3, 10)
        for _ in range(num_crops):
            bg = random.choice(backgrounds)
            crop_w = random.randint(100, 500)
            crop_h = random.randint(100, 400)
            bw, bh = bg.size
            if bw <= crop_w or bh <= crop_h:
                continue
            cx = random.randint(0, bw - crop_w)
            cy = random.randint(0, bh - crop_h)
            crop = bg.crop((cx, cy, cx + crop_w, cy + crop_h))
            px = random.randint(0, IMG_SIZE - crop_w)
            py = random.randint(0, IMG_SIZE - crop_h)
            canvas.paste(crop, (px, py))

    canvas = add_fake_squares(canvas, [])
    canvas = add_random_lines(canvas)
    canvas = add_noise_dots(canvas)

    canvas.save(img_path, "JPEG", quality=95)
    with open(label_path, "w") as f:
        f.write("")  # empty label = no objects


def generate_mixed_text_and_boards(boards, piece_sets, backgrounds, img_path, label_path):
    """Generate image with chess boards + heavy text (like a book page)."""
    num_boards = random.randint(1, 3)
    placements = place_boards(num_boards)

    canvas = random_background(backgrounds)
    draw = ImageDraw.Draw(canvas)
    all_labels = []

    # Heavy text FIRST (behind boards)
    num_blocks = random.randint(8, 20)
    for _ in range(num_blocks):
        font_size = random.randint(10, 30)
        try:
            font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", font_size)
        except:
            font = ImageFont.load_default()
        lines = []
        for _ in range(random.randint(2, 6)):
            words = random.sample(LOREM_WORDS + CHESS_WORDS, min(random.randint(3, 8), len(LOREM_WORDS)))
            lines.append(" ".join(words))
        text = "\n".join(lines)
        x = random.randint(0, IMG_SIZE - 200)
        y = random.randint(0, IMG_SIZE - 100)
        color = (random.randint(20, 100), random.randint(20, 100), random.randint(20, 100))
        draw.text((x, y), text, fill=color, font=font)

    # Place chess boards on top of text
    for (bx, by, size, _) in placements:
        board_img = random.choice(boards)
        pieces = random.choice(piece_sets)
        fen = generate_fen()
        rendered = render_board(board_img, pieces, fen, size)
        canvas.paste(rendered, (bx, by))
        all_labels.extend(get_labels(fen, bx, by, size))

    # More text around boards (captions, annotations)
    canvas = add_random_text(canvas, placements)

    canvas.save(img_path, "JPEG", quality=95)
    with open(label_path, "w") as f:
        f.write("\n".join(all_labels))


def generate_one(boards, piece_sets, backgrounds, img_path, label_path):
    """Generate one training image with 1-12 boards + noise."""
    num_boards = random.randint(1, MAX_BOARDS_PER_IMAGE)
    placements = place_boards(num_boards)

    canvas = random_background(backgrounds)
    all_labels = []

    # Add noise BEHIND boards (text, shapes that boards will cover)
    canvas = add_random_lines(canvas)
    canvas = add_random_shapes(canvas)

    # Place chess boards
    for (bx, by, size, _) in placements:
        board_img = random.choice(boards)
        pieces = random.choice(piece_sets)
        fen = generate_fen()

        rendered = render_board(board_img, pieces, fen, size)
        canvas.paste(rendered, (bx, by))
        all_labels.extend(get_labels(fen, bx, by, size))

    # Add fake squares (NOT chess boards — negative samples)
    canvas = add_fake_squares(canvas, placements)

    # Paste random real image crops (photos, icons, etc.)
    canvas = add_real_image_crops(canvas, placements, backgrounds)

    # Add noise ON TOP of everything (text labels, dots)
    canvas = add_random_text(canvas, placements)
    canvas = add_noise_dots(canvas)
    canvas = add_blur(canvas)

    canvas.save(img_path, "JPEG", quality=95)
    with open(label_path, "w") as f:
        f.write("\n".join(all_labels))


def main():
    print("Loading assets...")
    boards = [load_board(b) for b in os.listdir(BOARDS_DIR) if b.endswith(".png")]
    piece_sets = [load_pieces(p) for p in os.listdir(PIECES_DIR)
                  if os.path.isdir(f"{PIECES_DIR}/{p}")]
    backgrounds = load_backgrounds()
    print(f"Loaded {len(boards)} boards, {len(piece_sets)} piece sets, {len(backgrounds)} backgrounds")

    if not backgrounds:
        print("Tip: Run 'cd assets && python random_images_downloader2.py' to download background images")

    random.shuffle(boards)
    random.shuffle(piece_sets)
    split = int(len(piece_sets) * DATA_SPLIT)
    train_pieces = piece_sets[:split]
    val_pieces = piece_sets[split:]

    for split_name, pieces, count in [("train", train_pieces, 5000), ("val", val_pieces, 1000)]:
        img_dir = f"{DATASETS_DIR}/images/{split_name}"
        lbl_dir = f"{DATASETS_DIR}/labels/{split_name}"
        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)

        # Distribution:
        #   70% normal (boards + noise)
        #   10% text-only negative (no boards)
        #   10% images-only negative (no boards)
        #   10% text + boards (book-style)
        n_normal = int(count * 0.70)
        n_text_neg = int(count * 0.10)
        n_img_neg = int(count * 0.10)
        n_text_board = count - n_normal - n_text_neg - n_img_neg

        print(f"Generating {count} {split_name} images "
              f"({n_normal} normal, {n_text_neg} text-only, "
              f"{n_img_neg} image-only, {n_text_board} text+board)...")

        i = 0
        for _ in range(n_normal):
            generate_one(boards, pieces, backgrounds,
                         f"{img_dir}/{i:05d}.jpg", f"{lbl_dir}/{i:05d}.txt")
            i += 1
            if i % 500 == 0: print(f"  {i}/{count}")

        for _ in range(n_text_neg):
            generate_negative_text_only(backgrounds,
                                        f"{img_dir}/{i:05d}.jpg", f"{lbl_dir}/{i:05d}.txt")
            i += 1

        for _ in range(n_img_neg):
            generate_negative_images_only(backgrounds,
                                          f"{img_dir}/{i:05d}.jpg", f"{lbl_dir}/{i:05d}.txt")
            i += 1

        for _ in range(n_text_board):
            generate_mixed_text_and_boards(boards, pieces, backgrounds,
                                           f"{img_dir}/{i:05d}.jpg", f"{lbl_dir}/{i:05d}.txt")
            i += 1

        print(f"  {i}/{count} done")

    # Write dataset config
    with open("chess_detection_multi.yaml", "w") as f:
        f.write(f"""path: {os.path.abspath(DATASETS_DIR)}
train: images/train
val: images/val

nc: 13
names:
  - black_pawn
  - black_rook
  - black_knight
  - black_bishop
  - black_queen
  - black_king
  - white_pawn
  - white_rook
  - white_knight
  - white_bishop
  - white_queen
  - white_king
  - chess_board
""")

    print("Done! Dataset config: chess_detection_multi.yaml")


if __name__ == "__main__":
    main()



In [ ]:
# Generate data (skip if restored from Drive)
if not RESTORE_FROM_DRIVE:
    !python generate_multi_board.py

    # Backup to Drive
    print('\nBacking up data to Drive...')
    !tar czf /content/datasets_multi.tar.gz -C /content/datasets_multi .
    !cp /content/datasets_multi.tar.gz {DRIVE_DATA}
    !ls -lh {DRIVE_DATA}
    print('Data backed up!')
else:
    # Write yaml config for restored data
    with open('/content/chess_detection_multi.yaml', 'w') as f:
        f.write(f"""path: /content/datasets_multi\ntrain: images/train\nval: images/val\n\nnc: 13\nnames:\n  - black_pawn\n  - black_rook\n  - black_knight\n  - black_bishop\n  - black_queen\n  - black_king\n  - white_pawn\n  - white_rook\n  - white_knight\n  - white_bishop\n  - white_queen\n  - white_king\n  - chess_board\n""")
    print('Using restored data from Drive')

In [ ]:
# Verify data + show samples
import glob
from IPython.display import display
from PIL import Image

train_count = len(glob.glob(f'{LOCAL_DATA}/images/train/*.jpg'))
val_count = len(glob.glob(f'{LOCAL_DATA}/images/val/*.jpg'))
print(f'Train: {train_count}, Val: {val_count}')

for i in [0, 50, 200, 500, 1000]:
    try:
        display(Image.open(f'{LOCAL_DATA}/images/train/{i:05d}.jpg').resize((400, 400)))
    except: pass

---
## 3. Fine-tune

In [ ]:
from ultralytics import YOLO

# Check if resuming from previous run
resume_path = f'{DRIVE_WEIGHTS}/weights/last.pt'
if os.path.exists(resume_path):
    print(f'Resuming from {resume_path}')
    model = YOLO(resume_path)
    model.train(data='/content/chess_detection_multi.yaml', epochs=50, resume=True)
else:
    print('Starting fresh fine-tune from base_model.pt')
    model = YOLO('/content/2d-chess-pieces-detection/base_model.pt')
    results = model.train(
        data='/content/chess_detection_multi.yaml',
        epochs=25,
        imgsz=1280,
        batch=20,
        device=0,
        lr0=0.0001,
        optimizer='AdamW',
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        scale=0.5,
        degrees=5,
        project=DRIVE_WEIGHTS,
        name='.',
        exist_ok=True,
        plots=True,
    )
print('Training complete!')

---
## 4. Training Report

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Image as IPImage

# Find results
csv_candidates = [
    f'{DRIVE_WEIGHTS}/results.csv',
    f'{DRIVE_WEIGHTS}/./results.csv',
]
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)
if not csv_path:
    import glob
    csv_path = glob.glob(f'{DRIVE_WEIGHTS}/**/results.csv', recursive=True)[0]

df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Training Report - Multi-board + Noise', fontsize=16, fontweight='bold')

axes[0][0].plot(df['epoch'], df['train/box_loss'], label='Train', color='#1f77b4')
axes[0][0].plot(df['epoch'], df['val/box_loss'], label='Val', color='#ff7f0e')
axes[0][0].set_title('Box Loss'); axes[0][0].legend()

axes[0][1].plot(df['epoch'], df['train/cls_loss'], label='Train', color='#1f77b4')
axes[0][1].plot(df['epoch'], df['val/cls_loss'], label='Val', color='#ff7f0e')
axes[0][1].set_title('Classification Loss'); axes[0][1].legend()

axes[0][2].plot(df['epoch'], df['train/dfl_loss'], label='Train', color='#1f77b4')
axes[0][2].plot(df['epoch'], df['val/dfl_loss'], label='Val', color='#ff7f0e')
axes[0][2].set_title('DFL Loss'); axes[0][2].legend()

axes[1][0].plot(df['epoch'], df['metrics/mAP50(B)'], label='mAP50', color='#2ca02c')
axes[1][0].plot(df['epoch'], df['metrics/mAP50-95(B)'], label='mAP50-95', color='#d62728')
axes[1][0].set_title('mAP'); axes[1][0].legend()

axes[1][1].plot(df['epoch'], df['metrics/precision(B)'], label='Precision', color='#9467bd')
axes[1][1].plot(df['epoch'], df['metrics/recall(B)'], label='Recall', color='#8c564b')
axes[1][1].set_title('Precision & Recall'); axes[1][1].legend()

lr_cols = [c for c in df.columns if 'lr' in c]
for c in lr_cols: axes[1][2].plot(df['epoch'], df[c], label=c)
axes[1][2].set_title('Learning Rate'); axes[1][2].legend()

for ax in axes.flat: ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_WEIGHTS}/training_report.png', dpi=150)
plt.show()

# Print final metrics
last = df.iloc[-1]
print(f"\n{'='*50}")
print(f"FINAL RESULTS (Epoch {int(last['epoch'])})")
print(f"{'='*50}")
print(f"mAP50:     {last['metrics/mAP50(B)']:.4f}")
print(f"mAP50-95:  {last['metrics/mAP50-95(B)']:.4f}")
print(f"Precision: {last['metrics/precision(B)']:.4f}")
print(f"Recall:    {last['metrics/recall(B)']:.4f}")

# Show YOLO plots
plot_dir = os.path.dirname(csv_path)
for plot in ['confusion_matrix.png', 'confusion_matrix_normalized.png', 'labels.jpg']:
    p = f'{plot_dir}/{plot}'
    if os.path.exists(p):
        print(f'\n{plot}'); display(IPImage(filename=p, width=800))

---
## 5. Compare: Base Model vs Fine-tuned

In [ ]:
import time, numpy as np
from ultralytics import YOLO
from PIL import Image
from IPython.display import display

CLASS_NAMES = [
    "black_pawn","black_rook","black_knight","black_bishop","black_queen","black_king",
    "white_pawn","white_rook","white_knight","white_bishop","white_queen","white_king",
    "chess_board"
]
FEN_MAP = {
    'black_pawn':'p','black_rook':'r','black_knight':'n','black_bishop':'b','black_queen':'q','black_king':'k',
    'white_pawn':'P','white_rook':'R','white_knight':'N','white_bishop':'B','white_queen':'Q','white_king':'K',
}

def extract_fens(results):
    boards, pieces = [], []
    for box in results[0].boxes:
        cls, conf = int(box.cls[0]), float(box.conf[0])
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        name = CLASS_NAMES[cls]
        if name == 'chess_board': boards.append((x1,y1,x2,y2,conf))
        else: pieces.append((name,conf,(x1+x2)/2,(y1+y2)/2))
    boards.sort(key=lambda b: b[0])
    fens = []
    for bx1,by1,bx2,by2,_ in boards:
        sw,sh = (bx2-bx1)/8, (by2-by1)/8
        grid = [[None]*8 for _ in range(8)]
        for name,conf,cx,cy in pieces:
            if bx1<=cx<=bx2 and by1<=cy<=by2:
                c = min(7, max(0, int((cx-bx1)/sw)))
                r = min(7, max(0, int((cy-by1)/sh)))
                if grid[r][c] is None or conf > grid[r][c][1]:
                    grid[r][c] = (FEN_MAP[name], conf)
        rows = []
        for row in grid:
            s, e = '', 0
            for cell in row:
                if cell is None: e += 1
                else:
                    if e: s += str(e); e = 0
                    s += cell[0]
            if e: s += str(e)
            rows.append(s)
        fens.append('/'.join(rows))
    return len(boards), len(pieces), fens

# Load both models
best_pt = f'{DRIVE_WEIGHTS}/weights/best.pt'
base_pt = '/content/2d-chess-pieces-detection/base_model.pt'
model_base = YOLO(base_pt)
model_new = YOLO(best_pt)

# Test on val images
test_images = [f'{LOCAL_DATA}/images/val/{i:05d}.jpg' for i in [0, 10, 50, 100, 200]]

print(f"{'Image':>10s}  {'':>5s}  {'Boards':>8s}  {'Pieces':>8s}  {'Time':>10s}")
print('='*55)

for img_path in test_images:
    idx = os.path.basename(img_path).replace('.jpg','')

    t0 = time.time()
    res_base = model_base(img_path, conf=0.25, imgsz=1280, verbose=False)
    t_base = time.time() - t0
    b_base, p_base, f_base = extract_fens(res_base)

    t0 = time.time()
    res_new = model_new(img_path, conf=0.25, imgsz=1280, verbose=False)
    t_new = time.time() - t0
    b_new, p_new, f_new = extract_fens(res_new)

    print(f'{idx:>10s}  {"BASE":>5s}  {b_base:>8d}  {p_base:>8d}  {t_base*1000:>8.1f}ms')
    print(f'{"":>10s}  {"NEW":>5s}  {b_new:>8d}  {p_new:>8d}  {t_new*1000:>8.1f}ms')

    match = '\u2705' if f_base == f_new else '\u26a0\ufe0f'
    print(f'{"":>10s}  FEN match: {match}  (base={len(f_base)} boards, new={len(f_new)} boards)')
    print()

# Side-by-side visual comparison
print('\n--- Visual Comparison (first image) ---')
img_path = test_images[0]
res_base = model_base(img_path, conf=0.25, imgsz=1280, verbose=False)
res_new = model_new(img_path, conf=0.25, imgsz=1280, verbose=False)

ann_base = Image.fromarray(res_base[0].plot()[:,:,::-1])
ann_new = Image.fromarray(res_new[0].plot()[:,:,::-1])

# Combine side by side
w = max(ann_base.width, ann_new.width)
h = max(ann_base.height, ann_new.height)
combined = Image.new('RGB', (w*2 + 20, h + 40), (255, 255, 255))
combined.paste(ann_base.resize((w, h)), (0, 40))
combined.paste(ann_new.resize((w, h)), (w + 20, 40))
from PIL import ImageDraw, ImageFont
draw = ImageDraw.Draw(combined)
try: font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 24)
except: font = ImageFont.load_default()
draw.text((w//2 - 80, 8), 'BASE MODEL (v0.0.4)', fill=(255,0,0), font=font)
draw.text((w + 20 + w//2 - 80, 8), 'NEW MODEL (fine-tuned)', fill=(0,128,0), font=font)
display(combined.resize((combined.width//2, combined.height//2)))

In [ ]:
# Full validation comparison
print('Running full validation on both models...\n')

metrics_base = model_base.val(data='/content/chess_detection_multi.yaml', imgsz=1280, verbose=False)
metrics_new = model_new.val(data='/content/chess_detection_multi.yaml', imgsz=1280, verbose=False)

mb = metrics_base.results_dict
mn = metrics_new.results_dict

print(f"{'Metric':>20s}  {'Base (v0.0.4)':>15s}  {'New (fine-tuned)':>15s}  {'Diff':>10s}")
print('='*65)
for key in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']:
    label = key.replace('metrics/','').replace('(B)','')
    vb, vn = mb[key], mn[key]
    diff = vn - vb
    arrow = '\u2191' if diff > 0 else '\u2193' if diff < 0 else '='
    color = '' 
    print(f'{label:>20s}  {vb:>15.4f}  {vn:>15.4f}  {arrow} {abs(diff):>+.4f}')

# Save comparison report
report = f"""# Model Comparison Report\n
## Base Model (v0.0.4) vs Fine-tuned (multi-board + noise)\n
| Metric | Base | New | Diff |
|--------|------|-----|------|
"""
for key in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']:
    label = key.replace('metrics/','').replace('(B)','')
    vb, vn = mb[key], mn[key]
    report += f"| {label} | {vb:.4f} | {vn:.4f} | {vn-vb:+.4f} |\n"

with open(f'{DRIVE_WEIGHTS}/comparison_report.md', 'w') as f:
    f.write(report)
print(f'\nReport saved to {DRIVE_WEIGHTS}/comparison_report.md')

---
## 6. Export ONNX

In [ ]:
best = YOLO(f'{DRIVE_WEIGHTS}/weights/best.pt')
best.export(format='onnx', imgsz=1280, nms=True)

print(f'\nFiles on Drive:')
!ls -lh {DRIVE_WEIGHTS}/weights/best.pt {DRIVE_WEIGHTS}/weights/best.onnx
print(f'\nTraining report: {DRIVE_WEIGHTS}/training_report.png')
print(f'Comparison: {DRIVE_WEIGHTS}/comparison_report.md')

In [ ]:
# Optional: Download to local machine
from google.colab import files
files.download(f'{DRIVE_WEIGHTS}/weights/best.pt')
files.download(f'{DRIVE_WEIGHTS}/weights/best.onnx')